# CoT Generation for Nemotron Reasoning Challenge

This notebook generates Chain-of-Thought (CoT) reasoning data using LLM API
for the NVIDIA Nemotron Model Reasoning Challenge.

cloudwantyou@gmail.com
さん、asked me to how. So I posted here. \
Related \
https://www.kaggle.com/code/konbu17/nemotron-sft-lora-with-cot-v2-prep-now-plz-wait/comments


## Features
- **Type-specific prompting**: Special prompts for Text Encryption (77-word dictionary),
  Bit Manipulation (per-bit candidate analysis), and standard puzzles
- **Rule-based scoring**: Automated answer extraction and verification
- **CoT shortening**: Reduces oversized CoT to fit within token limits
- **Checkpoint/Resume**: Can resume from interruptions
- **Parallel API calls**: ThreadPoolExecutor for speed

## Requirements
- API access (DeepSeek or other Powerfull API OSS model)
- Please listen to his well-reasoned discussion. (https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/discussion/688360)
- Nemotron tokenizer (for token counting)

## Configuration

**IMPORTANT**: Replace the Azure API credentials with your own.

In [ ]:
import json
import os
import re
import time
import traceback
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import pandas as pd
from openai import AzureOpenAI
from transformers import AutoTokenizer

# ============================================================
# Config — Replace with your own Azure OpenAI credentials
# ============================================================
AZURE_ENDPOINT = "YOUR_AZURE_ENDPOINT"           # e.g., "https://your-resource.openai.azure.com/"
AZURE_API_KEY = "YOUR_AZURE_API_KEY"              # Your Azure OpenAI API key
AZURE_API_VERSION = "2025-04-01-preview"           # select an appropriate API version.
MODEL = "DeepSeek-V3.2"                            # Or your deployed model name
TEMPERATURE = 0.7
MAX_COMPLETION_TOKENS = 16384

# Input/Output paths — adjust for your environment
DATA_PATH = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv"
MODEL_PATH = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"  # For tokenizer only
OUTPUT_DIR = "/kaggle/working"
CHECKPOINT_DIR = "/kaggle/working/cot_checkpoint"

TOKEN_LIMIT = 7600
CHECKPOINT_INTERVAL = 50
MAX_WORKERS = 8

## Text Encryption: 77-word Dictionary

The Text Encryption puzzles use a monoalphabetic substitution cipher with a fixed 77-word vocabulary.

In [ ]:
WORD_LIST_77 = [
    "above", "alice", "ancient", "around", "beyond", "bird", "book", "bright",
    "castle", "cat", "cave", "chases", "clever", "colorful", "creates", "crystal",
    "curious", "dark", "discovers", "door", "dragon", "draws", "dreams", "explores",
    "follows", "forest", "found", "garden", "golden", "hatter", "hidden", "imagines",
    "in", "inside", "island", "key", "king", "knight", "library", "magical",
    "map", "message", "mirror", "mountain", "mouse", "mysterious", "near", "ocean",
    "palace", "potion", "princess", "puzzle", "queen", "rabbit", "reads", "school",
    "secret", "sees", "silver", "story", "strange", "student", "studies", "teacher",
    "the", "through", "tower", "treasure", "turtle", "under", "valley", "village",
    "watches", "wise", "wizard", "wonderland", "writes",
]

## Azure OpenAI Client & Tokenizer

In [ ]:
client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
)

print("Loading Nemotron tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)


def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

## Answer Extraction & Scoring

In [ ]:
def clean_boxed_content(text: str) -> str:
    text = re.sub(r'\\text\{[^}]*\}', '', text)
    text = text.replace('\\ ', ' ')
    text = re.sub(r'\\(?=\s)', '', text)
    text = ' '.join(text.split())
    return text.strip()


def extract_answer(response: str) -> str:
    if not isinstance(response, str) or not response:
        return ""
    think_end = response.find("</think>")
    after_think = response[think_end + len("</think>"):] if think_end != -1 else response

    boxed = re.findall(r'\\boxed\{([^}]*)\}', after_think)
    if boxed:
        return clean_boxed_content(boxed[-1])

    boxed = re.findall(r'\\boxed\{([^}]*)\}', response)
    if boxed:
        return clean_boxed_content(boxed[-1])

    if think_end != -1:
        text = after_think.strip()
        if text:
            return text.split('\n')[-1].strip()

    lines = [l.strip() for l in response.strip().split('\n') if l.strip()]
    return lines[-1] if lines else ""


def check_answer(pred: str, gt: str, puzzle_type: str) -> float:
    pred = pred.strip()
    gt = gt.strip()

    if puzzle_type == "Bit Manipulation":
        m = re.search(r'[01]{8}', pred)
        if m:
            pred = m.group(0)
        return 1.0 if pred == gt else 0.0

    elif puzzle_type in ("Gravitational Constant", "Unit Conversion"):
        try:
            pred_val = float(re.search(r'[\d.]+', pred).group(0))
            gt_val = float(gt)
            if abs(pred_val - gt_val) <= 0.05:
                return 1.0
            if abs(pred_val - gt_val) / max(abs(gt_val), 1e-9) <= 0.005:
                return 1.0
            return 0.0
        except (ValueError, AttributeError):
            return 0.0

    elif puzzle_type == "Numeral Conversion":
        pred_upper = pred.upper()
        gt_upper = gt.upper()
        m = re.search(r'[IVXLCDM]+', pred_upper)
        if m:
            pred_upper = m.group(0)
        return 1.0 if pred_upper == gt_upper else 0.0

    elif puzzle_type == "Text Encryption":
        return 1.0 if pred.lower() == gt.lower() else 0.0

    elif puzzle_type == "Equation Transformation":
        return 1.0 if pred == gt else 0.0

    else:
        return 1.0 if pred == gt else 0.0

## Text Encryption: Cipher Mapping Builder

In [ ]:
def build_cipher_mapping(prompt_text: str):
    lines = prompt_text.split('\n')
    plain_to_cipher = {}
    cipher_to_plain = {}
    for line in lines:
        if ' -> ' in line and 'Now, decrypt' not in line:
            parts = line.strip().split(' -> ')
            if len(parts) == 2:
                encrypted_words = parts[0].strip().split()
                plain_words = parts[1].strip().split()
                if len(encrypted_words) == len(plain_words):
                    for ew, pw in zip(encrypted_words, plain_words):
                        for ec, pc in zip(ew, pw):
                            cipher_to_plain[ec] = pc
                            plain_to_cipher[pc] = ec
    return plain_to_cipher, cipher_to_plain


def build_te_dictionary_str(prompt_text: str) -> str:
    p2c, _ = build_cipher_mapping(prompt_text)
    lines = []
    for word in WORD_LIST_77:
        encrypted = ''.join(p2c.get(c, '?') for c in word)
        lines.append(f"{encrypted} -> {word}")
    return '\n'.join(lines)

## Bit Manipulation: Per-bit Candidate Analysis

Each output bit is independently determined by a boolean function of 1-3 input bits.
We pre-compute all candidate functions for each bit position.

In [ ]:
def parse_bm_examples(prompt_text: str):
    lines = prompt_text.strip().split('\n')
    examples = []
    query = None
    for line in lines:
        line = line.strip()
        if ' -> ' in line and 'Now,' not in line:
            parts = line.split(' -> ')
            if len(parts) == 2:
                inp = parts[0].strip()
                out = parts[1].strip()
                if len(inp) == 8 and len(out) == 8 and all(c in '01' for c in inp + out):
                    examples.append((inp, out))
        elif 'determine the output for:' in line.lower():
            m = re.search(r'[01]{8}', line)
            if m:
                query = m.group(0)
    return examples, query


def analyze_bm_candidates(examples, query):
    n = len(examples)
    inputs = [[int(ex[0][b]) for b in range(8)] for ex in examples]
    outputs = [[int(ex[1][b]) for b in range(8)] for ex in examples]
    query_bits = [int(query[b]) for b in range(8)]

    ops_1 = {"ID": lambda a: a, "NOT": lambda a: 1 - a}
    ops_2 = {
        "AND": lambda a, b: a & b, "OR": lambda a, b: a | b, "XOR": lambda a, b: a ^ b,
        "NAND": lambda a, b: 1 - (a & b), "NOR": lambda a, b: 1 - (a | b), "XNOR": lambda a, b: 1 - (a ^ b),
    }

    per_bit_info = []
    for obit in range(8):
        target = [outputs[e][obit] for e in range(n)]
        candidates = []

        for pos in range(8):
            src = [inputs[e][pos] for e in range(n)]
            for name, func in ops_1.items():
                if all(func(src[e]) == target[e] for e in range(n)):
                    candidates.append((name, [pos], func(query_bits[pos])))

        for p1 in range(8):
            for p2 in range(p1 + 1, 8):
                s1 = [inputs[e][p1] for e in range(n)]
                s2 = [inputs[e][p2] for e in range(n)]
                for name, func in ops_2.items():
                    if all(func(s1[e], s2[e]) == target[e] for e in range(n)):
                        candidates.append((name, [p1, p2], func(query_bits[p1], query_bits[p2])))

        if candidates:
            pred_values = set(c[2] for c in candidates)
            if len(pred_values) == 1:
                status = "CERTAIN"
                predicted = list(pred_values)[0]
            else:
                status = "AMBIGUOUS"
                votes_0 = sum(1 for c in candidates if c[2] == 0)
                votes_1 = sum(1 for c in candidates if c[2] == 1)
                predicted = 1 if votes_1 > votes_0 else 0
        else:
            status = "NO_CANDIDATES"
            predicted = 0

        per_bit_info.append({
            "obit": obit, "candidates": candidates,
            "status": status, "predicted": predicted, "n_candidates": len(candidates),
        })
    return per_bit_info


def build_bm_analysis_str(prompt_text: str) -> str:
    examples, query = parse_bm_examples(prompt_text)
    if not examples or not query:
        return ""

    analysis = analyze_bm_candidates(examples, query)
    lines = [f"Query input: {query}", f"Number of examples: {len(examples)}", ""]
    lines.append("Per-bit analysis (output bit 0 = leftmost):")

    for info in analysis:
        obit = info["obit"]
        status = info["status"]
        predicted = info["predicted"]
        n_cands = info["n_candidates"]

        if status == "CERTAIN":
            cands = info["candidates"]
            best = min(cands, key=lambda c: len(c[1]))
            lines.append(f"  Bit {obit}: {status} -> {predicted}  "
                         f"(best: {best[0]}(input[{','.join(str(p) for p in best[1])}]), "
                         f"{n_cands} candidates all agree)")
        elif status == "AMBIGUOUS":
            cands = info["candidates"]
            votes_0 = sum(1 for c in cands if c[2] == 0)
            votes_1 = sum(1 for c in cands if c[2] == 1)
            lines.append(f"  Bit {obit}: {status} (votes: 0->{votes_0}, 1->{votes_1})")
            for c in [c for c in cands if c[2] == 0][:2]:
                lines.append(f"    predicts 0: {c[0]}(input[{','.join(str(p) for p in c[1])}])")
            for c in [c for c in cands if c[2] == 1][:2]:
                lines.append(f"    predicts 1: {c[0]}(input[{','.join(str(p) for p in c[1])}])")
        else:
            lines.append(f"  Bit {obit}: {status} (may need 3-input/composite function)")

    shifts = []
    for info in analysis:
        if info["status"] == "CERTAIN":
            best = min(info["candidates"], key=lambda c: len(c[1]))
            if len(best[1]) == 1:
                shifts.append((best[1][0] - info["obit"]) % 8)
    if shifts:
        shift_counts = Counter(shifts)
        most_common = shift_counts.most_common(1)[0]
        lines.append(f"\nDetected shift pattern: most common shift = {most_common[0]} "
                     f"(appears {most_common[1]}/{len(shifts)} times)")

    predicted_bits = ''.join(str(info["predicted"]) for info in analysis)
    certain_count = sum(1 for info in analysis if info["status"] == "CERTAIN")
    lines.append(f"\nPredicted output (best guess): {predicted_bits}")
    lines.append(f"Certain bits: {certain_count}/8, Ambiguous bits: {8 - certain_count}/8")
    return '\n'.join(lines)

## Prompt Construction (Type-specific)

In [ ]:
def build_gpt_prompt_standard(prompt_text: str) -> list:
    system_msg = (
        "You are an expert puzzle solver. You will be given a puzzle from Alice's Wonderland.\n"
        "You MUST show detailed step-by-step reasoning before giving your answer.\n\n"
        "Your response format:\n"
        "1. First, analyze each example carefully to identify the pattern/rule\n"
        "2. Show your work: calculations, comparisons, hypothesis testing\n"
        "3. Verify your rule against ALL given examples\n"
        "4. Apply the rule to the new input, showing each step\n"
        "5. Put your final answer inside \\boxed{}\n\n"
        "IMPORTANT: Do NOT skip steps. Show ALL intermediate calculations.\n"
        "IMPORTANT: Your answer inside \\boxed{} must be ONLY the answer value, no units or extra text."
    )
    user_msg = prompt_text + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    return [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]


def build_gpt_prompt_text_encryption(prompt_text: str) -> list:
    dictionary_str = build_te_dictionary_str(prompt_text)
    system_msg = (
        "You are an expert at solving monoalphabetic substitution ciphers.\n"
        "In Alice's Wonderland, text is encrypted using a secret letter-by-letter substitution cipher.\n"
        "The vocabulary consists of exactly 77 known English words.\n\n"
        "You MUST show detailed step-by-step reasoning:\n"
        "1. Build the character mapping (cipher->plain) from the given examples\n"
        "2. List all character mappings you've found\n"
        "3. For each encrypted word in the target, try to decrypt it using the mapping\n"
        "4. If some characters are unknown, use the 77-word dictionary to find the matching word\n"
        "5. Give the final decrypted text\n\n"
        "IMPORTANT: Your answer inside \\boxed{} must be ONLY the plain text words separated by spaces, "
        "with NO backslashes or LaTeX formatting.\n"
        "For example: \\boxed{queen sees secret}"
    )
    user_msg = (
        f"{prompt_text}\n\n"
        f"Here is the complete 77-word dictionary with their encrypted forms based on the cipher mapping:\n"
        f"{dictionary_str}\n\n"
        f"Please put your final answer inside `\\boxed{{}}`. For example: `\\boxed{{your answer}}`"
    )
    return [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]


def build_gpt_prompt_bit_manipulation(prompt_text: str) -> list:
    analysis_str = build_bm_analysis_str(prompt_text)
    system_msg = (
        "You are an expert at solving bit manipulation puzzles.\n\n"
        "KEY INSIGHT: Each of the 8 output bits is independently determined by a boolean function "
        "of 1-3 input bits. The puzzle is essentially 8 independent boolean function identification problems.\n\n"
        "Known statistics about these puzzles:\n"
        "- Operation distribution: ID/copy (40%), AND (15%), XOR (8%), XNOR (7%), OR (7%), "
        "NOR (6%), NOT (5%), NAND (3%), 3-input (2%), composite (6%)\n"
        "- Shift pattern: output bit i typically reads from input bit (i+shift)%8 where shift is constant. "
        "shift=0 almost never occurs (0.5%).\n"
        "- 59% of puzzles use at most 2 types of operations (e.g., ID*5 + XOR*3)\n\n"
        "Solving strategy:\n"
        "1. For each output bit (0-7, left to right), find which input bits and boolean function "
        "produce the correct output for ALL examples\n"
        "2. If only one candidate function works, the bit is CERTAIN\n"
        "3. If multiple candidates disagree, use heuristics (prefer simpler ops, consistent shift)\n"
        "4. A pre-computed analysis of candidates is provided below to help you\n\n"
        "IMPORTANT: Your answer inside \\boxed{} must be ONLY the 8-bit binary string."
    )
    user_msg = (
        f"{prompt_text}\n\n"
        f"=== Pre-computed per-bit candidate analysis ===\n"
        f"{analysis_str}\n\n"
        f"Using the analysis above, determine the correct 8-bit output. "
        f"Show your reasoning for each bit, then give the final 8-bit answer.\n\n"
        f"Please put your final answer inside `\\boxed{{}}`. For example: `\\boxed{{10110100}}`"
    )
    return [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]


def build_gpt_messages(prompt_text: str, puzzle_type: str) -> list:
    if puzzle_type == "Text Encryption":
        return build_gpt_prompt_text_encryption(prompt_text)
    elif puzzle_type == "Bit Manipulation":
        return build_gpt_prompt_bit_manipulation(prompt_text)
    else:
        return build_gpt_prompt_standard(prompt_text)

## API Call & CoT Shortening

In [ ]:
def call_gpt(messages: list, max_tokens: int = MAX_COMPLETION_TOKENS, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                messages=messages,
                max_completion_tokens=max_tokens,
                model=MODEL,
                temperature=TEMPERATURE,
            )
            return response.choices[0].message.content or ""
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
            else:
                print(f"  API FAILED after {retries} retries: {e}")
    return ""


def shorten_cot(prompt_text: str, cot: str, overflow_ratio: float) -> str:
    target_reduction_pct = int(overflow_ratio * 100)
    target_reduction_pct = min(target_reduction_pct + 10, 70)
    system_msg = (
        "You are a text editor that shortens chain-of-thought reasoning while preserving correctness. "
        f"You MUST reduce the length by approximately {target_reduction_pct}%. "
        "Keep the final answer in \\boxed{} exactly the same. "
        "Output ONLY the shortened reasoning followed by the \\boxed{} answer."
    )
    user_msg = f"Original puzzle:\n{prompt_text}\n\nOriginal reasoning (needs ~{target_reduction_pct}% reduction):\n{cot}"
    messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]
    result = call_gpt(messages, max_tokens=MAX_COMPLETION_TOKENS)
    return result if result else cot

## Checkpoint Management

In [ ]:
def save_checkpoint(results: list, checkpoint_path: str):
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    with open(checkpoint_path, 'w') as f:
        json.dump(results, f)


def load_checkpoint(checkpoint_path: str) -> list:
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r') as f:
            return json.load(f)
    return []

## Process Single Row

In [ ]:
def process_row(row_data: dict) -> dict:
    """Process a single puzzle row. Thread-safe."""
    row_id = row_data["id"]
    puzzle_type = row_data["type"]
    prompt_text = row_data["prompt"]
    gt_answer = str(row_data["answer"])

    try:
        messages = build_gpt_messages(prompt_text, puzzle_type)
        gpt_response = call_gpt(messages)

        if not gpt_response:
            return {
                "id": row_id, "prompt": prompt_text, "answer": gt_answer,
                "type": puzzle_type, "generated_cot": "", "extracted_answer": "",
                "score": 0.0, "cot_tokens": 0,
            }

        extracted = extract_answer(gpt_response)
        score = check_answer(extracted, gt_answer, puzzle_type)
        token_count = count_tokens(gpt_response)

        return {
            "id": row_id, "prompt": prompt_text, "answer": gt_answer,
            "type": puzzle_type, "generated_cot": gpt_response,
            "extracted_answer": extracted, "score": score, "cot_tokens": token_count,
        }
    except Exception as e:
        print(f"  ERROR processing {row_id}: {e}")
        traceback.print_exc()
        return {
            "id": row_id, "prompt": prompt_text, "answer": gt_answer,
            "type": puzzle_type, "generated_cot": "", "extracted_answer": "",
            "score": 0.0, "cot_tokens": 0,
        }

## Main Execution

Load data, process all rows with parallel API calls, save results.

In [ ]:
# Configuration
RESUME = True  # Set to True to resume from checkpoint
WORKERS = MAX_WORKERS

checkpoint_path = os.path.join(CHECKPOINT_DIR, "results_checkpoint.json")

print(f"Loading data from {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print(f"Total rows: {len(df)}")

# Resume from checkpoint
existing_results = []
processed_ids = set()
if RESUME:
    existing_results = load_checkpoint(checkpoint_path)
    processed_ids = {r["id"] for r in existing_results}
    print(f"Resuming: {len(existing_results)} already processed")

# Filter to unprocessed
remaining = df[~df['id'].isin(processed_ids)]
print(f"Remaining: {len(remaining)} rows")

In [ ]:
if len(remaining) == 0:
    print("All rows already processed!")
    results = existing_results
else:
    results = list(existing_results)
    print_lock = Lock()
    start_time = time.time()
    row_dicts = remaining.to_dict('records')
    completed = 0
    total = len(row_dicts)
    type_stats = defaultdict(lambda: {"total": 0, "correct": 0})

    for r in existing_results:
        type_stats[r["type"]]["total"] += 1
        if r.get("score", 0) >= 1.0:
            type_stats[r["type"]]["correct"] += 1

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        future_to_row = {executor.submit(process_row, rd): rd for rd in row_dicts}
        for future in as_completed(future_to_row):
            result = future.result()
            results.append(result)
            completed += 1
            type_stats[result["type"]]["total"] += 1
            if result.get("score", 0) >= 1.0:
                type_stats[result["type"]]["correct"] += 1
            elapsed = time.time() - start_time
            rate = completed / elapsed if elapsed > 0 else 0
            eta = (total - completed) / rate if rate > 0 else 0
            if completed % 10 == 0 or completed == total:
                with print_lock:
                    score_str = "OK" if result["score"] >= 1.0 else "NG"
                    print(f"[{len(existing_results)+completed}/{len(df)}] "
                          f"{score_str} {result['type'][:4]} id={result['id'][:8]} "
                          f"tokens={result.get('cot_tokens', 0):>5} "
                          f"| {rate:.1f}/s ETA={eta/60:.0f}m")
            if completed % CHECKPOINT_INTERVAL == 0:
                save_checkpoint(results, checkpoint_path)

    save_checkpoint(results, checkpoint_path)

## Results Summary

In [ ]:
print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

type_stats = defaultdict(lambda: {"total": 0, "correct": 0})
for r in results:
    type_stats[r["type"]]["total"] += 1
    if r.get("score", 0) >= 1.0:
        type_stats[r["type"]]["correct"] += 1

total_correct = 0
total_count = 0
for t, stats in sorted(type_stats.items()):
    acc = stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"  {t}: {stats['correct']}/{stats['total']} ({acc:.1%})")
    total_correct += stats["correct"]
    total_count += stats["total"]

overall_acc = total_correct / total_count if total_count > 0 else 0
print(f"  OVERALL: {total_correct}/{total_count} ({overall_acc:.1%})")

token_counts = [r["cot_tokens"] for r in results if r.get("cot_tokens", 0) > 0]
if token_counts:
    print(f"\nCoT Token Stats: min={min(token_counts)}, max={max(token_counts)}, "
          f"mean={sum(token_counts)/len(token_counts):.0f}, "
          f"over_{TOKEN_LIMIT}={sum(1 for t in token_counts if t > TOKEN_LIMIT)}")

## Save Results

In [ ]:
results_df = pd.DataFrame(results)

# All results (correct + incorrect)
tf_path = os.path.join(OUTPUT_DIR, "train_split_with_cot_tf.csv")
results_df[["id", "prompt", "answer", "type", "generated_cot"]].to_csv(tf_path, index=False)
print(f"Saved all results: {tf_path} ({len(results_df)} rows)")

# Correct only
correct_df = results_df[results_df["score"] >= 1.0].copy()
t_only_path = os.path.join(OUTPUT_DIR, "train_split_with_cot_t_only.csv")
correct_df[["id", "prompt", "answer", "type", "generated_cot"]].to_csv(t_only_path, index=False)
print(f"Saved correct only: {t_only_path} ({len(correct_df)} rows)")

print("\nDone!")